# Data Preprocessing Pipeline (Section 4.1)
This notebook implements the preprocessing workflow used before coverage and annotation audits.

The goal is to take source datasets with different schemas and annotation structures and map them into one comparable, post-level format that can be inspected, merged, deduplicated, and exported.

The notebook is designed to be reproducible and rerunnable with two modes:
- **Primary mode**: HateXplain + MHS
- **Extended mode**: HateXplain + MHS + ElSherief (optional toggle)

### Output schema produced by harmonization
Every dataset is mapped to a common format:
- `post_id`: stable identifier for the post/comment when available
- `text`: normalized text content used in downstream analysis
- `raw_label`: original label payload kept for traceability
- `binary_hate`: comparable binary hate label used for audits
- `targets`: normalized list of targeted groups
- `dataset`: dataset provenance tag
- `text_dedup_key`: normalized text key used for fallback deduplication

### Important design choices
- HateXplain is already one row per post, but it stores multiple annotators inside each row.
- MHS is annotator-level in the raw parquet, so this notebook collapses it to one row per post before combining it with HateXplain.
- Targets are preserved as list-like data in memory and only serialized to pipe-delimited strings at TSV export time.
- Deduplication happens after harmonization so all datasets are compared under the same schema.

### Pipeline stages
1. Configure paths, thresholds, and output locations
2. Load local data, with MHS cached locally when possible
3. Harmonize labels and target columns into a shared schema
4. Standardize to comparable post-level rows
5. Union and deduplicate the resulting corpora
6. Run threshold sensitivity summaries
7. Validate target coverage
8. Save outputs for downstream analysis

In [1]:
from __future__ import annotations

# Core utilities for configuration and path handling
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional

# Parsing utilities
import json
import re

# Dataframe operations
import pandas as pd

In [ ]:
WORKDIR = Path('/Users/RevaH/Documents/COS534/benchmarking_dogwhistles')

@dataclass
class PipelineConfig:
    # `workdir` is the anchor used to resolve all relative paths in the notebook.
    workdir: Path = WORKDIR

    # Local sources expected to exist in the repository.
    hatexplain_path: Path = Path('data/hatexplain.json')
    mhs_local_path: Path = Path('data/measuring_hate_speech.parquet')
    elsherief_path: Optional[Path] = Path('data/implicit-hate-corpus/implicit_hate_v1_stg3_posts.tsv')

    # Remote MHS source used only if the local parquet is missing or a refresh is requested.
    mhs_hf_uri: str = 'hf://datasets/ucberkeley-dlab/measuring-hate-speech/data/train-00000-of-00001.parquet'
    refresh_mhs_local_copy: bool = False

    # Main preprocessing controls.
    include_elsherief: bool = True
    mhs_primary_threshold: float = 0.5
    sensitivity_thresholds: List[float] = field(default_factory=lambda: [0.4, 0.5, 0.6])

    # Directory where notebook exports are written.
    output_dir: Path = Path('outputs/preprocessing')


cfg = PipelineConfig()
# Convert all configured relative paths into absolute paths once up front.
cfg.output_dir = cfg.workdir / cfg.output_dir
cfg.mhs_local_path = cfg.workdir / cfg.mhs_local_path
cfg.hatexplain_path = cfg.workdir / cfg.hatexplain_path
cfg.elsherief_path = cfg.workdir / cfg.elsherief_path if cfg.elsherief_path else None

# Create output/cache directories early so later cells can assume they exist.
cfg.output_dir.mkdir(parents=True, exist_ok=True)
cfg.mhs_local_path.parent.mkdir(parents=True, exist_ok=True)
cfg

PipelineConfig(workdir=PosixPath('/Users/RevaH/Documents/COS534/benchmarking_dogwhistles'), hatexplain_path=PosixPath('/Users/RevaH/Documents/COS534/benchmarking_dogwhistles/data/hatexplain.json'), mhs_local_path=PosixPath('/Users/RevaH/Documents/COS534/benchmarking_dogwhistles/data/measuring_hate_speech.parquet'), elsherief_path=PosixPath('/Users/RevaH/Documents/COS534/benchmarking_dogwhistles/data/implicit-hate-corpus/implicit_hate_v1_stg3_posts.tsv'), mhs_hf_uri='hf://datasets/ucberkeley-dlab/measuring-hate-speech/data/train-00000-of-00001.parquet', refresh_mhs_local_copy=False, include_elsherief=False, mhs_primary_threshold=0.5, sensitivity_thresholds=[0.4, 0.5, 0.6], output_dir=PosixPath('/Users/RevaH/Documents/COS534/benchmarking_dogwhistles/outputs/preprocessing'))

## Load helpers
These helpers make file loading robust and reproducible:
- resolve relative paths against the project directory,
- parse HateXplain JSON variants,
- use a local MHS parquet cache when available,
- fallback to one-time remote download for MHS when cache is missing.

In [3]:
def resolve_path(path: Path) -> Path:
    """Resolve absolute/relative paths against the configured project root."""
    p = Path(path)
    if p.is_absolute():
        return p
    return (cfg.workdir / p).resolve()


def read_hatexplain_json(path: Path) -> pd.DataFrame:
    """Read HateXplain JSON from either list-of-records or dict-of-records format."""
    path = resolve_path(path)
    if not path.exists():
        raise FileNotFoundError(f'Missing HateXplain file: {path}')

    with open(path, 'r', encoding='utf-8') as f:
        obj = json.load(f)

    if isinstance(obj, list):
        rows = obj
    elif isinstance(obj, dict):
        rows = []
        for k, v in obj.items():
            if isinstance(v, dict):
                row = v.copy()
                row.setdefault('post_id', k)
                rows.append(row)
    else:
        raise ValueError('Unsupported HateXplain JSON structure.')

    return pd.DataFrame(rows)


def get_mhs_dataframe(local_path: Path, hf_uri: str, refresh_local: bool = False) -> pd.DataFrame:
    """Load local MHS parquet; if missing (or refresh requested), fetch and cache locally."""
    local_path = resolve_path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)

    if local_path.exists() and not refresh_local:
        print(f'Loading MHS from local copy: {local_path}')
        return pd.read_parquet(local_path)

    print('Fetching MHS from Hugging Face URI...')
    df = pd.read_parquet(hf_uri)
    df.to_parquet(local_path, index=False)
    print(f'Saved local MHS copy to: {local_path}')
    return df


def read_any_table(path: Path) -> pd.DataFrame:
    """Generic reader for csv/parquet/json/jsonl/tsv files."""
    path = resolve_path(path)
    if not path.exists():
        raise FileNotFoundError(f'Missing file: {path}')

    suffix = path.suffix.lower()
    if suffix == '.csv':
        return pd.read_csv(path)
    if suffix == '.tsv':
        return pd.read_csv(path, sep='\t')
    if suffix in {'.parquet', '.pq'}:
        return pd.read_parquet(path)
    if suffix in {'.json', '.jsonl'}:
        try:
            return pd.read_json(path, lines=True)
        except ValueError:
            return pd.read_json(path)
    raise ValueError(f'Unsupported file type: {suffix} for {path}')

## Schema mapping (adjust if needed)
This mapping specifies which raw columns should be used for each dataset.

Update this section if your local files use different column names.
HateXplain and MHS are then converted to the shared schema used in all downstream steps.

In [4]:
SCHEMA = {
    'hatexplain': {
        'id_col': 'post_id',
        'text_col': 'post_tokens',  # fallback to text if needed
        'label_col': 'annotators',
        'targets_col': 'annotators',
    },
    'mhs': {
        'id_col': 'comment_id',
        'text_col': 'text',
        'score_col': 'hate_speech_score',
        # For MHS we derive targets from multi-hot target_* columns when present.
        'targets_col': 'target',
    },
    'elsherief': {
        'id_col': 'ID',
        'text_col': 'post',
        'label_col': 'class',
        'targets_col': 'target',
    },
}

In [ ]:
def normalize_text_for_dedup(text: str) -> str:
    """Create a normalized text key used for fallback deduplication."""
    if not isinstance(text, str):
        return ''
    return re.sub(r'\s+', ' ', text.strip().lower())


def is_truthy(value) -> bool:
    """Interpret booleans across mixed dtypes (bool/int/float/string)."""
    if value is None or pd.isna(value):
        return False
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, float)):
        return value != 0
    s = str(value).strip().lower()
    return s in {'1', 'true', 't', 'yes', 'y'}


def normalize_target_token(token: str) -> Optional[str]:
    """Normalize target labels to a consistent lowercase, human-readable token."""
    if token is None:
        return None
    s = str(token).strip().lower()
    if not s or s in {'none', 'null', 'nan'}:
        return None
    s = s.replace('_', ' ')
    return s


def parse_targets_generic(value) -> List[str]:
    """Normalize targets into a clean list[str] regardless of raw storage format."""
    if value is None or pd.isna(value):
        return []

    values = []
    if isinstance(value, list):
        # Already list-like, which is the ideal in-memory representation.
        values = value
    elif isinstance(value, str):
        # Handle strings that may store a single target, a delimited set, or a JSON-like list.
        s = value.strip()
        if not s:
            return []
        if s.startswith('[') and s.endswith(']'):
            try:
                parsed = json.loads(s.replace("'", '"'))
                if isinstance(parsed, list):
                    values = parsed
                else:
                    values = [parsed]
            except Exception:
                values = [s]
        elif '|' in s:
            values = [x.strip() for x in s.split('|')]
        elif ',' in s:
            values = [x.strip() for x in s.split(',')]
        else:
            values = [s]
    else:
        values = [value]

    out = []
    for v in values:
        norm = normalize_target_token(v)
        if norm:
            out.append(norm)
    return sorted(set(out))


def label_to_binary(label_value) -> Optional[int]:
    """Map heterogeneous label strings to binary hate labels used by the audit."""
    if label_value is None or pd.isna(label_value):
        return None
    s = str(label_value).strip().lower()
    if s in {'hate', 'hateful', 'implicit_hate', 'explicit_hate', '1', 'true', 'yes'}:
        return 1
    if s in {'normal', 'non-hate', 'non_hate', 'offensive', 'not_hate', '0', 'false', 'no'}:
        return 0
    return None


def extract_mhs_targets_from_row(row: pd.Series, target_cols: List[str]) -> List[str]:
    """Collapse MHS multi-hot target_* columns into a single target list."""
    targets = []
    for col in target_cols:
        if not is_truthy(row.get(col)):
            continue

        suffix = col[len('target_'):]
        axis = None
        group = suffix
        if '_' in suffix:
            axis, group = suffix.split('_', 1)

        # Example: `target_race_black` becomes `black`.
        # For `*_other`, keep the axis so the signal is not lost.
        token = f'{axis} other' if (group == 'other' and axis) else group
        norm = normalize_target_token(token)
        if norm:
            targets.append(norm)
    return sorted(set(targets))

In [ ]:
def aggregate_mhs_to_post_level(mhs_harmonized: pd.DataFrame, threshold: float) -> pd.DataFrame:
    """Collapse annotator-level MHS rows to one row per post_id/text."""
    required = {'post_id', 'text', 'raw_label', 'targets', 'dataset', 'text_dedup_key'}
    missing = required - set(mhs_harmonized.columns)
    if missing:
        raise KeyError(f'MHS harmonized dataframe missing columns: {sorted(missing)}')

    base = mhs_harmonized.copy()
    # MHS scores arrive per annotator; convert to numeric before aggregating.
    base['raw_label'] = pd.to_numeric(base['raw_label'], errors='coerce')

    grouped = (
        base.groupby(['post_id', 'text', 'dataset', 'text_dedup_key'], dropna=False)
        .agg(
            # Average the annotator-level hate score into one post-level score.
            raw_label=('raw_label', 'mean'),
            # Keep the number of annotations so the output still exposes how much evidence a post had.
            n_annotations=('raw_label', 'size'),
            # Union all annotator targets observed for the post.
            targets=('targets', lambda s: sorted(set(t for v in s for t in (v if isinstance(v, list) else [])))),
        )
        .reset_index()
    )
    # Apply the same threshold after aggregation so MHS becomes comparable to one-row-per-post datasets.
    grouped['binary_hate'] = (grouped['raw_label'] >= threshold).astype('Int64')

    # Keep the same column order as the harmonized schema and add n_annotations as metadata.
    cols = ['post_id', 'text', 'raw_label', 'binary_hate', 'targets', 'dataset', 'text_dedup_key', 'n_annotations']
    return grouped[cols]


def serialize_targets_for_tsv(value) -> str:
    """Serialize list-like targets to a compact pipe-delimited string for TSV output."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ''
    if isinstance(value, list):
        return '|'.join(str(x) for x in value)
    return str(value)


def serialize_any_for_tsv(value) -> str:
    """Serialize nested values so TSV exports remain parseable and stable."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ''
    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False)
    return str(value)


def dataframe_for_tsv(df: pd.DataFrame) -> pd.DataFrame:
    """Create a copy with list/dict columns converted to string-safe TSV representations."""
    out = df.copy()
    # Keep list-like targets readable in spreadsheets while still easy to split later.
    if 'targets' in out.columns:
        out['targets'] = out['targets'].apply(serialize_targets_for_tsv)
    # Preserve raw nested payloads in a loss-limited string form for inspection/debugging.
    if 'raw_label' in out.columns:
        out['raw_label'] = out['raw_label'].apply(serialize_any_for_tsv)
    return out

In [ ]:
def parse_targets_generic(value) -> List[str]:
    """Normalize targets into a clean list[str] regardless of raw storage format."""
    if value is None:
        return []

    values = []
    if isinstance(value, list):
        values = value
    elif isinstance(value, tuple):
        values = list(value)
    elif isinstance(value, str):
        # This redefinition is intentional: it is more defensive than the earlier helper
        # and safely handles list-like values that would otherwise cause ambiguous `pd.isna` checks.
        s = value.strip()
        if not s:
            return []
        if s.startswith('[') and s.endswith(']'):
            try:
                parsed = json.loads(s.replace("'", '"'))
                if isinstance(parsed, list):
                    values = parsed
                else:
                    values = [parsed]
            except Exception:
                values = [s]
        elif '|' in s:
            values = [x.strip() for x in s.split('|')]
        elif ',' in s:
            values = [x.strip() for x in s.split(',')]
        else:
            values = [s]
    else:
        try:
            if pd.isna(value):
                return []
        except Exception:
            pass
        values = [value]

    out = []
    for v in values:
        norm = normalize_target_token(v)
        if norm:
            out.append(norm)
    return sorted(set(out))

## Harmonization
These functions transform each dataset into a common dataframe format.

Important behavior:
- MHS is binarized using the configured threshold.
- HateXplain supports `text` or `post_tokens` for content.
- Labels are normalized into `binary_hate` when possible.
- Targets are parsed into list format for consistent downstream use.

In [ ]:
def harmonize_hatexplain(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize HateXplain rows into the shared schema used for analysis."""
    out = pd.DataFrame()

    # HateXplain usually stores a stable `post_id`, but fall back gracefully if a variant omits it.
    if 'post_id' in df.columns:
        out['post_id'] = df['post_id'].astype(str)
    elif 'id' in df.columns:
        out['post_id'] = df['id'].astype(str)
    else:
        out['post_id'] = pd.Series(df.index).astype(str)

    # Some HateXplain exports store already-joined text; others store token lists.
    if 'text' in df.columns:
        out['text'] = df['text'].astype(str)
    elif 'post_tokens' in df.columns:
        out['text'] = df['post_tokens'].apply(lambda x: ' '.join(x) if isinstance(x, list) else str(x))
    else:
        raise KeyError('HateXplain text not found. Expected text or post_tokens.')

    def ann_to_binary(annotators) -> Optional[int]:
        if not isinstance(annotators, list):
            return None
        vals = []
        for a in annotators:
            if isinstance(a, dict):
                b = label_to_binary(a.get('label'))
                if b is not None:
                    vals.append(b)
        if not vals:
            return None
        # Reduce multiple annotator labels to one post-level decision by majority vote.
        return int(sum(vals) >= (len(vals) / 2.0))

    def ann_to_targets(annotators) -> List[str]:
        if not isinstance(annotators, list):
            return []
        all_targets = []
        for a in annotators:
            if isinstance(a, dict):
                all_targets.extend(parse_targets_generic(a.get('target')))
        # Keep the union of all target groups mentioned by any annotator.
        return sorted(set(all_targets))

    if 'label' in df.columns:
        out['raw_label'] = df['label']
        out['binary_hate'] = df['label'].apply(label_to_binary).astype('Int64')
    elif 'annotators' in df.columns:
        # Preserve the full annotator payload in `raw_label` so it can be inspected later.
        out['raw_label'] = df['annotators']
        out['binary_hate'] = df['annotators'].apply(ann_to_binary).astype('Int64')
    else:
        out['raw_label'] = None
        out['binary_hate'] = pd.Series([pd.NA] * len(df), dtype='Int64')

    if 'targets' in df.columns:
        out['targets'] = df['targets'].apply(parse_targets_generic)
    elif 'annotators' in df.columns:
        out['targets'] = df['annotators'].apply(ann_to_targets)
    else:
        out['targets'] = [[] for _ in range(len(df))]

    out['dataset'] = 'hatexplain'
    out['text_dedup_key'] = out['text'].apply(normalize_text_for_dedup)
    return out


def harmonize_mhs(df: pd.DataFrame, threshold: float) -> pd.DataFrame:
    """Standardize MHS rows, binarize score, and collapse multi-hot targets."""
    m = SCHEMA['mhs']
    required = [m['text_col'], m['score_col']]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f'MHS missing columns: {missing}. Available columns: {list(df.columns)}')

    out = pd.DataFrame()
    out['post_id'] = df[m['id_col']].astype(str) if m['id_col'] in df.columns else pd.Series(df.index).astype(str)
    out['text'] = df[m['text_col']].astype(str)
    out['raw_label'] = pd.to_numeric(df[m['score_col']], errors='coerce')
    # At this stage the threshold is still applied per annotation row; aggregation happens later.
    out['binary_hate'] = (out['raw_label'] >= threshold).astype('Int64')

    # Prefer explicit multi-hot target columns because they are the richest MHS target signal.
    multi_hot_cols = [
        c for c in df.columns
        if c.startswith('target_') and re.match(r'^target_[^_]+_.+', c) is not None
    ]
    if multi_hot_cols:
        out['targets'] = df.apply(lambda r: extract_mhs_targets_from_row(r, multi_hot_cols), axis=1)
    elif m['targets_col'] in df.columns:
        out['targets'] = df[m['targets_col']].apply(parse_targets_generic)
    else:
        out['targets'] = [[] for _ in range(len(df))]

    out['dataset'] = 'mhs'
    out['text_dedup_key'] = out['text'].apply(normalize_text_for_dedup)
    return out


def harmonize_elsherief(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize ElSherief implicit-hate rows into the shared schema."""
    s = SCHEMA['elsherief']
    out = pd.DataFrame()

    out['post_id'] = df[s['id_col']].astype(str) if s['id_col'] in df.columns else pd.Series(df.index).astype(str)
    out['text'] = df[s['text_col']].astype(str) if s['text_col'] in df.columns else pd.Series([''] * len(df))

    if s['label_col'] in df.columns:
        out['raw_label'] = df[s['label_col']]
        out['binary_hate'] = df[s['label_col']].apply(label_to_binary).astype('Int64')
    else:
        # For target/implied_statement files with no explicit class label, assume implicit hate examples.
        out['raw_label'] = 'implicit_hate'
        out['binary_hate'] = pd.Series([1] * len(df), dtype='Int64')

    if s['targets_col'] in df.columns:
        out['targets'] = df[s['targets_col']].apply(parse_targets_generic)
    else:
        out['targets'] = [[] for _ in range(len(df))]

    out['dataset'] = 'elsherief'
    out['text_dedup_key'] = out['text'].apply(normalize_text_for_dedup)
    return out


def harmonize_generic(df: pd.DataFrame, dataset_name: str, threshold: float = 0.5) -> pd.DataFrame:
    """Dispatch to dataset-specific harmonizers and keep one common output schema."""
    if dataset_name == 'hatexplain':
        return harmonize_hatexplain(df)
    if dataset_name == 'mhs':
        return harmonize_mhs(df, threshold=threshold)
    if dataset_name == 'elsherief':
        return harmonize_elsherief(df)

    raise ValueError(f'Unsupported dataset_name: {dataset_name}')

## Run once (primary threshold)
This block executes the main preprocessing flow at `cfg.mhs_primary_threshold`.

What it does:
- loads raw HateXplain and MHS data,
- harmonizes both into the shared schema,
- optionally appends ElSherief if enabled,
- unions rows and performs ID-first then text-key deduplication,
- prints summary counts for sanity checks.

In [ ]:
# 1) Load raw inputs.
# These stay unchanged so the notebook can always reconstruct harmonized outputs from source data.
hx_raw = read_hatexplain_json(cfg.hatexplain_path)
mhs_raw = get_mhs_dataframe(cfg.mhs_local_path, cfg.mhs_hf_uri, refresh_local=cfg.refresh_mhs_local_copy)

# 2) Harmonize each dataset into the same in-memory schema.
# At this point MHS is still annotator-level, while HateXplain is already post-level.
hx = harmonize_generic(hx_raw, 'hatexplain', threshold=cfg.mhs_primary_threshold)
mhs = harmonize_generic(mhs_raw, 'mhs', threshold=cfg.mhs_primary_threshold)

# 3) Standardize granularity so both datasets become one row per post.
# HateXplain already behaves that way; MHS must be collapsed across annotator rows.
hx_std = hx.drop_duplicates(subset=['post_id'], keep='first').copy()
mhs_std = aggregate_mhs_to_post_level(mhs, threshold=cfg.mhs_primary_threshold)
frames = [hx_std, mhs_std]

# 4) Optional third dataset branch.
# This is kept separate so the default primary pipeline remains just HateXplain + MHS.
if cfg.include_elsherief:
    els_raw = read_any_table(cfg.elsherief_path)
    els = harmonize_generic(els_raw, 'elsherief', threshold=cfg.mhs_primary_threshold)
    els_std = els.drop_duplicates(subset=['post_id'], keep='first').copy()
    frames.append(els_std)

# 5) Union the standardized datasets.
union_df = pd.concat(frames, ignore_index=True)

# 6) Deduplicate in two passes.
# First prefer explicit post IDs, then use normalized text as a fallback for rows without usable IDs.
with_id = union_df[union_df['post_id'].notna() & (union_df['post_id'] != '')].drop_duplicates(subset=['post_id'], keep='first')
without_id = union_df[~union_df.index.isin(with_id.index)].drop_duplicates(subset=['text_dedup_key'], keep='first')
dedup_df = pd.concat([with_id, without_id], ignore_index=True).drop_duplicates(subset=['text_dedup_key'], keep='first')

# 7) Quick sanity checks.
# These counts are the fastest way to verify that harmonization and aggregation behaved as expected.
print('HateXplain raw rows:', len(hx_raw))
print('MHS raw rows (annotator-level):', len(mhs_raw))
print('HateXplain standardized rows (post-level):', len(hx_std))
print('MHS standardized rows (post-level):', len(mhs_std))
print('Union rows:', len(union_df))
print('Dedup rows:', len(dedup_df))
display(dedup_df.head(3))

Loading MHS from local copy: /Users/RevaH/Documents/COS534/benchmarking_dogwhistles/data/measuring_hate_speech.parquet
HateXplain raw rows: 20148
MHS raw rows (annotator-level): 135556
HateXplain standardized rows (post-level): 20148
MHS standardized rows (post-level): 39565
Union rows: 59713
Dedup rows: 59621


,post_id,text,raw_label,binary_hate,targets,dataset,text_dedup_key,n_annotations
0,1179055004553900032_twitter,i dont think im getting my baby them white 9 h...,"[{'label': 'normal', 'annotator_id': 1, 'targe...",0,[],hatexplain,i dont think im getting my baby them white 9 h...,NaN
1,1179063826874032128_twitter,we cannot continue calling ourselves feminists...,"[{'label': 'normal', 'annotator_id': 1, 'targe...",0,[],hatexplain,we cannot continue calling ourselves feminists...,NaN
2,1178793830532956161_twitter,nawt yall niggers ignoring me,"[{'label': 'normal', 'annotator_id': 4, 'targe...",0,[african],hatexplain,nawt yall niggers ignoring me,NaN


## Sensitivity analysis (configurable thresholds)
This section reruns preprocessing summaries for each threshold in `cfg.sensitivity_thresholds`.

Use it to check whether core corpus-level outcomes change materially around the primary cutoff (0.5).

In [12]:
def run_preprocess_summary(threshold: float, include_elsherief: bool) -> Dict[str, float]:
    """Return compact corpus-level stats for a given MHS threshold."""
    # Rebuild harmonized tables at this threshold.
    hx_local = harmonize_generic(hx_raw, 'hatexplain', threshold=threshold)
    mhs_local = harmonize_generic(mhs_raw, 'mhs', threshold=threshold)

    # Enforce one-row-per-post comparability across datasets.
    hx_local = hx_local.drop_duplicates(subset=['post_id'], keep='first').copy()
    mhs_local = aggregate_mhs_to_post_level(mhs_local, threshold=threshold)
    parts = [hx_local, mhs_local]

    if include_elsherief:
        els_raw_local = read_any_table(cfg.elsherief_path)
        els_local = harmonize_generic(els_raw_local, 'elsherief', threshold=threshold)
        els_local = els_local.drop_duplicates(subset=['post_id'], keep='first').copy()
        parts.append(els_local)

    all_df = pd.concat(parts, ignore_index=True)
    d = all_df.drop_duplicates(subset=['post_id', 'text_dedup_key'], keep='first')

    return {
        'threshold': threshold,
        'include_elsherief': include_elsherief,
        'rows_union': len(all_df),
        'rows_dedup': len(d),
        'hate_rate': float(pd.to_numeric(d['binary_hate'], errors='coerce').mean()) if len(d) else float('nan')
    }


sensitivity_rows = [run_preprocess_summary(float(t), cfg.include_elsherief) for t in cfg.sensitivity_thresholds]
sensitivity_df = pd.DataFrame(sensitivity_rows)
display(sensitivity_df)

,threshold,include_elsherief,rows_union,rows_dedup,hate_rate
0,0.4,False,59713,59713,0.197012
1,0.5,False,59713,59713,0.183902
2,0.6,False,59713,59713,0.170740


In [ ]:
# Target integrity checks (post-level exports)
# This cell is meant to answer one question: after all harmonization and aggregation,
# did we keep the target information we expected to keep?
print('Target integrity checks:')

# HateXplain: inspect target coverage and examples.
# This is mostly a qualitative sanity check because target data is nested inside annotators.
hx_nonempty = hx_std['targets'].apply(lambda x: isinstance(x, list) and len(x) > 0)
print('HateXplain rows:', len(hx_std))
print('HateXplain rows with non-empty targets:', int(hx_nonempty.sum()), f'({hx_nonempty.mean():.2%})')
print('HateXplain sample targets:', hx_std.loc[hx_nonempty, 'targets'].head(10).tolist())

# MHS: compare raw target_* evidence against standardized post-level targets.
# If a post has any target flag in the raw annotator rows, the standardized row should not be empty.
mhs_target_cols = [
    c for c in mhs_raw.columns
    if c.startswith('target_') and re.match(r'^target_[^_]+_.+', c) is not None
]
raw_any_target = (
    mhs_raw.assign(_row_has_target=mhs_raw[mhs_target_cols].apply(lambda r: any(is_truthy(v) for v in r), axis=1))
    .groupby('comment_id')['_row_has_target']
    .max()
)
raw_any_target.index = raw_any_target.index.astype(str)
std_has_target = mhs_std.set_index('post_id')['targets'].apply(lambda x: isinstance(x, list) and len(x) > 0)
std_has_target.index = std_has_target.index.astype(str)

common_ids = raw_any_target.index.intersection(std_has_target.index)
comparison = pd.DataFrame({
    'raw_any_target': raw_any_target.loc[common_ids],
    'std_has_target': std_has_target.loc[common_ids],
})

print('MHS rows:', len(mhs_std))
print('MHS rows with non-empty targets:', int(std_has_target.sum()), f'({std_has_target.mean():.2%})')
print('MHS agreement table (raw any-target vs standardized non-empty target):')
print(comparison.value_counts().to_string())

missed = comparison[(comparison['raw_any_target']) & (~comparison['std_has_target'])]
print('MHS posts with raw targets but empty standardized targets:', len(missed))

# Hard assertions turn this from a visual check into a regression guard.
assert len(common_ids) == len(mhs_std), 'Mismatch between standardized MHS post_ids and raw comment_ids.'
assert len(missed) == 0, 'Some MHS posts lost target annotations during standardization.'

Target integrity checks:
HateXplain rows: 20148
HateXplain rows with non-empty targets: 16230 (80.55%)
HateXplain sample targets: [['african'], ['asian'], ['caucasian', 'women'], ['jewish'], ['african'], ['african', 'homosexual', 'jewish'], ['african', 'jewish'], ['african'], ['islam'], ['african']]
MHS rows: 39565
MHS rows with non-empty targets: 39563 (99.99%)
MHS agreement table (raw any-target vs standardized non-empty target):
raw_any_target  std_has_target
True            True              39563
False           False                 2
MHS posts with raw targets but empty standardized targets: 0


## Save outputs
Exports the primary union, deduplicated corpus, and sensitivity summary as CSVs under `outputs/preprocessing/`.

In [21]:
# Ensure output directory exists.
# This makes the save cell safe to rerun even in a clean checkout.
cfg.output_dir.mkdir(parents=True, exist_ok=True)

# Prepare TSV-safe views.
# In-memory dataframes keep lists/dicts because they are easier to analyze,
# but TSV needs plain strings to remain readable and importable.
hx_std_tsv = dataframe_for_tsv(hx_std)
mhs_std_tsv = dataframe_for_tsv(mhs_std)
union_tsv = dataframe_for_tsv(union_df)
dedup_tsv = dataframe_for_tsv(dedup_df)
sensitivity_tsv = dataframe_for_tsv(sensitivity_df)

# Save per-dataset standardized exports so each source can also be inspected on its own.
hx_std_tsv.to_csv(cfg.output_dir / 'hatexplain_standardized.tsv', sep='\t', index=False)
mhs_std_tsv.to_csv(cfg.output_dir / 'mhs_standardized.tsv', sep='\t', index=False)

# Save combined artifacts used by downstream analysis.
dedup_tsv.to_csv(cfg.output_dir / 'dedup_primary.tsv', sep='\t', index=False)
union_tsv.to_csv(cfg.output_dir / 'union_primary.tsv', sep='\t', index=False)
sensitivity_tsv.to_csv(cfg.output_dir / 'sensitivity_summary.tsv', sep='\t', index=False)

print('Saved TSV outputs:')
print('-', cfg.output_dir / 'hatexplain_standardized.tsv')
print('-', cfg.output_dir / 'mhs_standardized.tsv')
print('-', cfg.output_dir / 'dedup_primary.tsv')
print('-', cfg.output_dir / 'union_primary.tsv')
print('-', cfg.output_dir / 'sensitivity_summary.tsv')

Saved TSV outputs:
- /Users/RevaH/Documents/COS534/benchmarking_dogwhistles/outputs/preprocessing/hatexplain_standardized.tsv
- /Users/RevaH/Documents/COS534/benchmarking_dogwhistles/outputs/preprocessing/mhs_standardized.tsv
- /Users/RevaH/Documents/COS534/benchmarking_dogwhistles/outputs/preprocessing/dedup_primary.tsv
- /Users/RevaH/Documents/COS534/benchmarking_dogwhistles/outputs/preprocessing/union_primary.tsv
- /Users/RevaH/Documents/COS534/benchmarking_dogwhistles/outputs/preprocessing/sensitivity_summary.tsv


## Rerun with ElSherief
To include ElSherief in the same pipeline:
1. Place the local file and update `cfg.elsherief_path` if needed.
2. Set `cfg.include_elsherief = True`.
3. Rerun from the primary run section onward to regenerate all outputs.